# Consistency relations

Calculate the tensor running of running with N2LO corrections (`n=2`, `p=2`).
Install SRI with `pip install -e .` and select that environment in your notebook editor.

Formulas are grouped by slow-roll weight. Displayed decimals use six significant
digits; `digits=None` shows all stored digits. Returned expressions and exports
keep their original precision.

In [1]:
from pathlib import Path
import sys

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "utils/notebook_display.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook inside the SRI repository.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from utils.notebook_display import show_equation, show_spectra, show_consistency

import sympy as sp
from IPython.display import display
from sri import consistency, ConsistencyOrder
from sri.presentation import simplify_expression
from sri import serialization

sp.init_printing()


## Orders and options

- `n=2`: tensor running of running; `p=2`: N2LO corrections.
- `method="iterative"` (default) or `"closed"`: background elimination.
- `symbolic=False` (default) or `True`: numerical or exact coefficients.
- `kinflation=False` (default) or `True`: canonical inflation or variable sound speed.
- `coefficient_tolerance=1e-10`: numerical coefficient convergence only.

In [2]:
p = 2
n = 2
method = "iterative"
symbolic = False
kinflation = False
coefficient_tolerance = 1e-10

options = dict(method=method, symbolic=symbolic, kinflation=kinflation,
               coefficient_tolerance=coefficient_tolerance)


## Generate the consistency relation

The absolute cutoff is `W=n+1+p`, with remainder at weight `W+1`.
`components` contains the contribution at each retained weight.

In [3]:
relation = consistency(n=n, p=p, **options)
show_consistency(relation)
print("n, p, W, remainder weight:", relation.n, relation.p, relation.W, relation.error_order)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

n, p, W, remainder weight: 2 2 5 6


`ConsistencyOrder.resolve` infers any missing index from the other two using
`W=n+1+p`. `consistency` accepts the same compatible pairs directly.

In [4]:
order = ConsistencyOrder.resolve(n=n, W=n + 1 + p)
assert (order.n, order.p, order.W) == (relation.n, relation.p, relation.W)
print(order)


ConsistencyOrder(n=2, p=2, W=5)


## Compare the algebra methods

Generate exact relations with `symbolic=True` using both methods and check their equality.

In [5]:
exact_relation = {m: consistency(n=n, p=p, method=m, symbolic=True, kinflation=kinflation)
                  for m in ("iterative", "closed")}
residual = sp.expand(exact_relation["iterative"].expression
                     - exact_relation["closed"].expression)
assert residual == 0
print("Difference between methods:", residual)


Difference between methods: 0


## Simplify

`simplify_expression(..., exact=True)` collects exact factors. Display the relation
before and after simplification and check algebraic equality. With numerical
coefficients, `exact=False` also permits simple-fraction recognition.

In [6]:
original = exact_relation[method].expression
simplified = simplify_expression(original, exact=True)
print("Before simplify:")
show_equation(r"\alpha_{" + str(n) + r"}^t", original)
print("After simplify:")
show_equation(r"\alpha_{" + str(n) + r"}^t", simplified.expression)
assert sp.expand(original - simplified.expression) == 0
print(simplified.metadata)


Before simplify:


<IPython.core.display.Math object>

After simplify:


<IPython.core.display.Math object>

{'recognized_fraction_count': 0, 'certifies_exact_coefficients': True, 'method': 'exact factor collection', 'numerical_recognition': False}


## Numerical evaluation

`scalar=[n_s-1,alpha_s,beta_s,...]` lists the scalar spectral derivatives.
Supply every derivative appearing in the selected tensor relation.

In [7]:
scalar = [-0.035, 0.001, 0.0001] + [0.0] * relation.W
sound = [0.002, 0.0001] + [0.0] * relation.W
sound_inputs = dict(cs=0.7, sound=sound) if kinflation else {}
print(relation.evaluate(r=0.02, scalar=scalar, **sound_inputs))


5.291273715835331e-07


## Variable scalar sound speed

`kinflation=True` selects k-inflation. Supply positive `cs` and
`sound=[alpha_0^cs,alpha_1^cs,...]`: `alpha_0^cs=d ln(c_s)/d ln(k)`;
higher entries are its successive logarithmic scale derivatives.

In [8]:
relation_cs = consistency(n=n, p=p, method=method, symbolic=symbolic, kinflation=True,
                         coefficient_tolerance=coefficient_tolerance)
print(relation_cs.evaluate(r=0.02, scalar=scalar, cs=0.7, sound=sound))
print("Model:", relation_cs.model, "Method:", relation_cs.method)


2.1710021775766823e-06
Model: kinflation Method: iterative


## LaTeX and YAML

`relation.latex()` includes the formal remainder. The commented file-writing
lines save an export only when explicitly enabled.

In [9]:
latex_relation = relation.latex()
yaml_text = serialization.dumps({
    "model": relation.model, "method": relation.method,
    "n": relation.n, "p": relation.p, "W": relation.W,
    "coefficients": relation.coefficient_metadata,
    "expression": str(relation.expression),
    "components": {str(weight): str(expression) for weight, expression in relation.components},
})
# from pathlib import Path
# Path("consistency.yaml").write_text(yaml_text)
# Path("relation.tex").write_text(latex_relation)


## Corresponding CLI options

| Python | CLI |
| --- | --- |
| `n=2, p=2` | `-n 2 -p 2` |
| `ConsistencyOrder.resolve(n=2, W=5)` | `-n 2 -W 5` |
| `method="iterative"` (default) | No algebra flag (default) |
| `method="closed"` | `--closed` |
| `symbolic=False` (default) | No coefficient flag (default) |
| `symbolic=True` | `--symbolic` |
| `kinflation=True` | `--kinflation` |
| `coefficient_tolerance=1e-10` | `--coefficient-tolerance 1e-10` |
| `simplify_expression(...)` | `--simplify` |
| LaTeX / YAML and explicit file writing | `--format latex` / `yaml`, `--output PATH` |

The CLI also returns spectra when requesting a relation. The Python example
here calculates the selected relation; spectra have [their own notebook](01_spectra.ipynb).
See the [API guide](../docs/api.md) for parameter conventions.